# PhysAR-IDS -- NB6b (v6): did the fully adaptive attacker against the pipeline converge?

**Why.** NB6 stopped the fully adaptive white-box search (the one that maximises `min(margin gap, (tau - error)/tau)`) after 100 rounds. Where rows were still improving at that point the evasion rates are lower bounds: 8% of rows for the robust model at 1.0 sigma, and **45% for the NB5 defended model at 1.0 sigma** -- roughly the share of ransomware and reverseshell rows, whose reported pipeline evasion of 0.000 is therefore not established. This notebook gives the attacker **300 rounds** at 0.5 and 1.0 sigma against both Layer-1 models plus the saved gate, and reports the evasion curve per class.

**Setup.** The five saved gates (`ae_gate_v6_seed{s}.pt`) and Layer-1 models are reloaded (nothing retrained; the gate threshold and clean false-positive rate are checked against NB6). Rows: a fixed random 500 per attack class (a subset of NB6's rows). Final decisions are made by the CPU classifier and the gate. 0.5 sigma is new here (NB6 had 0.1 / 0.3 / 1.0).

**Read it as:** if the curves are flat and no rows are still improving, the NB6 numbers stand; if ransomware / reverseshell climb, the "gate stops them" statement must be withdrawn. The attack is still greedy, so the final numbers remain lower bounds.

**Needs:** `ae_gate_v6_seed{s}.pt`, `xgb_robust_v6_seed{s}.json`, `xgb_defended_v6_seed{s}.json`, `attack3_seed{s}_v6.pkl` (NB6) and the data files. **Runtime:** several hours (about 1.5 h per seed).

## 0. Setup

In [ ]:
import os, sys, random, time, json, hashlib, shutil, platform, warnings, copy
import numpy as np, pandas as pd, joblib
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset
import xgboost as xgb
from scipy import stats
from sklearn.metrics import f1_score
import matplotlib; import matplotlib.pyplot as plt
import gymnasium as gym
from gymnasium import spaces
try:
    from sb3_contrib import RecurrentPPO
    from stable_baselines3.common.callbacks import CheckpointCallback
except ImportError as e:
    raise ImportError('Install in a terminal (not in a cell): pip install stable-baselines3 sb3-contrib gymnasium --no-deps') from e
warnings.filterwarnings('ignore')

BASE_DIR    = r'C:\Users\user\Desktop\smartrid1'
PROJECT_DIR = os.path.join(BASE_DIR, 'SmartGrid_Dissertation')
os.makedirs(PROJECT_DIR, exist_ok=True); os.chdir(PROJECT_DIR)

PIPELINE_VERSION = 'v6'
SEEDS = [42, 7, 123, 2024, 99]
SMOKE_TEST = False

# ---- PPO / environment settings (unchanged from the reference design) ----
EPS_BOUND = 0.3; MAX_STEPS = 10
EVASION_REWARD = 10.0; STEP_PENALTY_SCALE = 0.01; TIMEOUT_PENALTY = -1.0
TOTAL_TIMESTEPS = 50000; CKPT_FREQ = 5000
EPS_TOTAL_BUDGET = 0.1
CLIP_BUDGETS = {'1.0': 1.0, 'unconstrained': None}     # budgets at which agents are RETRAINED with range clipping (stage B)
SAVED_AGENTS = {'0.1': ('ppo_attack2_budgeted_{v}_seed{s}', 0.1), '0.2': ('ppo_attack2_budget0p2_{v}_seed{s}', 0.2), '0.3': ('ppo_attack2_budget0p3_{v}_seed{s}', 0.3),
                '0.5': ('ppo_attack2_budget0p5_{v}_seed{s}', 0.5), '1.0': ('ppo_attack2_budget1p0_{v}_seed{s}', 1.0), 'unconstrained': ('ppo_attack2_unconstrained_{v}_seed{s}', None)}
EVAL_CHUNK = 8192


# ---- Defence 1 (adversarial training) settings ----
PGD_EPS = 0.1; PGD_ITERS = 10                       # as NB3 / NB3b (all 93 features, alpha = eps / 4)
TRAIN_PPO_TAGS = ['0.1', '0.3']                     # PPO agents (NB4 / NB4b, trained against the STANDARD model) whose examples go into the training set
EVAL_PPO_TAGS = ['0.1', '0.3', '0.5', '1.0', 'unconstrained']   # budgets 0.5 / 1.0 / unconstrained were never seen in training
N_PPO_TRAIN_PER_CLASS = 8000
AUG_RATIO = 1.0                                     # adversarial training rows per clean row (the paper's 1:1)
# ---- white-box (tree-greedy, NB3c / NB3d) attack against both models ----
K_CAND = 12; MAX_ITERS = 60; CHUNK = 20000; BLOCK = 300_000
WB_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '0.5': 0.5, '1.0': 1.0}
N_WB_PER_CLASS = 2000                               # attack rows per class searched (same rows for every seed, model and budget)


# ---- NB6: Attack 3 -- adaptive attacker against the WHOLE pipeline (Layer 1 classifier + Layer 2 autoencoder gate) ----
LAYER1 = {'rob': 'xgb_robust_{v}_seed{s}.json', 'def': 'xgb_defended_{v}_seed{s}.json'}      # NB5b robust model (primary) and NB5 defended model
AW_BUDGETS = {'0.1': 0.1, '0.3': 0.3, '1.0': 1.0}      # sigma, cyber features only (same attacker-controlled set as NB3c / NB5)
N_AW_PER_CLASS = 1000                                    # rows per attack class (a random subset of NB5's 2,000 per class)
MAX_ITERS_AWARE = 100                                    # rounds for the greedy search against the pipeline
GATE_SCALE = 1.0                                         # weight of the gate term in the joint score min(margin gap, GATE_SCALE * (tau - err) / tau)
N_FOLDS = 5; GATE_EPOCHS = 30; GATE_PCTL = 95; GATE_HIDDEN = 32; GATE_BOTTLENECK = 12; GATE_CYBER_BOOST = 1.0
RUN_PPO_AWARE = True; PPO_AWARE_MODELS = ['rob']; PPO_AWARE_BUDGETS = {'0.3': 0.3, '1.0': 1.0}      # black-box PPO retrained against the pipeline's hard-label verdict
PPO_UNAWARE_TAGS = ['0.3', '1.0', 'unconstrained']      # the saved NB4 agents (trained against the standard classifier, gate-unaware)


# ---- NB6b: convergence of the fully adaptive white-box attack against the pipeline ----
LONG_ITERS = 300                       # NB6 used MAX_ITERS_AWARE = 100
CHECKPOINTS = [100, 150, 200, 300]
LONG_BUDGETS = {'0.5': 0.5, '1.0': 1.0}
N_LONG_PER_CLASS = 500                 # random subset of NB6's 1,000 rows per class (halves the cost; same rows for every seed, model and budget)

os.environ['PYTHONHASHSEED'] = '42'
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
with warnings.catch_warnings(record=True) as _caught:
    warnings.simplefilter('always')
    _m = xgb.XGBClassifier(tree_method='hist', device='cuda', n_estimators=2); _m.fit(np.zeros((10, 2)), np.array([0, 1] * 5))
    _cpu = any('No visible GPU' in str(w.message) for w in _caught)
BACKEND = {'tree_method': 'hist'} if _cpu else {'tree_method': 'hist', 'device': 'cuda'}
BACKEND_NAME = 'cpu(hist)' if _cpu else 'gpu(cuda,hist)'
print(f'torch {torch.__version__} ({DEVICE}) | xgboost {xgb.__version__} fit backend: {BACKEND_NAME}')
import sb3_contrib, stable_baselines3
print(f'sb3-contrib {sb3_contrib.__version__} | stable-baselines3 {stable_baselines3.__version__} | gymnasium {gym.__version__}')
print('Working directory:', os.getcwd())

## 1. Load v6 data

In [ ]:
data = joblib.load(f'phase1_data_{PIPELINE_VERSION}.pkl')
assert data.get('PIPELINE_VERSION') == PIPELINE_VERSION and data.get('TEST_FRAC') == 0.15, 'version mismatch'
X_train = data['X_train_scaled'].astype(np.float32); X_test = data['X_test_scaled'].astype(np.float32)
y_train = np.asarray(data['y_train']); y_test = np.asarray(data['y_test'])
FEATURE_COLS = list(data['FEATURE_COLS']); CYBER_IDX = np.asarray(data['CYBER_IDX']); N_FEAT = X_train.shape[1]; N_CYBER = len(CYBER_IDX)
class_names = ['benign', 'backdoor', 'Bruteforce', 'FDI', 'ransomware', 'reverseshell']
best_params = json.load(open(f'xgb_tuned_best_params_{PIPELINE_VERSION}.json'))
COMMON = dict(objective='multi:softprob', num_class=6, eval_metric='mlogloss', n_jobs=-1, **BACKEND)

atk_tr = y_train != 0; X_atk_train, y_atk_train = X_train[atk_tr], y_train[atk_tr]
atk_te = y_test != 0;  X_atk_test,  y_atk_test  = X_test[atk_te],  y_test[atk_te]
test_burst = pd.factorize(pd.MultiIndex.from_arrays([y_test, np.asarray(data['PLCID_test']), np.asarray(data['timestamp_test'])]))[0][atk_te]
FEAT_LO, FEAT_HI = X_train.min(0), X_train.max(0)
print(f'Features {N_FEAT} | attacker-controlled (cyber) {N_CYBER} | training pool {X_atk_train.shape} | evaluation rows {X_atk_test.shape} in {len(np.unique(test_burst))} attack bursts')
print('Training pool class counts:', np.bincount(y_atk_train)[1:], '| evaluation class counts:', np.bincount(y_atk_test)[1:])
nb2 = joblib.load(f'optuna_xgboost_results_{PIPELINE_VERSION}.pkl')
nb2_tuned = np.array([nb2['results']['tuned'][s]['test_f1'] for s in SEEDS if s in nb2['results']['tuned']])
print(f'NB2 tuned test F1 over seeds: {nb2_tuned.mean():.4f} +/- {nb2_tuned.std(ddof=1):.4f}')

## 2. White-box attack (with per-round trace)

In [ ]:
def get_thresholds(clf):
    """feature index -> sorted unique float32 split thresholds used anywhere in the ensemble."""
    df = clf.get_booster().trees_to_dataframe(); sp = df[df['Feature'] != 'Leaf']
    return {int(str(f).lstrip('f')): np.unique(g['Split'].to_numpy(dtype=np.float32)) for f, g in sp.groupby('Feature')}

class Margin:
    """Raw per-class scores of a fitted classifier (the white-box access). Scores in blocks and, if the GPU runs out of memory, halves the block and retries."""
    def __init__(self, clf): self.b = clf.get_booster(); self.bs = BLOCK
    def _pred(self, X):
        while True:
            try:
                return [np.asarray(self.b.inplace_predict(np.ascontiguousarray(X[s:s + self.bs], dtype=np.float32), predict_type='margin', validate_features=False)) for s in range(0, len(X), self.bs)]
            except xgb.core.XGBoostError as e:
                if 'out of memory' not in str(e).lower() or self.bs <= 20000: raise
                self.bs = max(20000, self.bs // 2); print(f'      GPU out of memory -> scoring in blocks of {self.bs:,} rows', flush=True)
    def __call__(self, X):
        out = self._pred(X); return np.concatenate(out) if len(out) > 1 else out[0]
def gap_of(z): return (z[:, 0] - z[:, 1:].max(1)).astype(np.float64)       # > 0  <=>  benign is the arg-max class

def interval(o, j, B):
    """allowed interval of feature j for rows with original values o: L-inf ball (if B) intersected with the training range (never further out than the row already is)."""
    lo_r, hi_r = np.minimum(FEAT_LO[j], o), np.maximum(FEAT_HI[j], o)
    if B is None: return lo_r.astype(np.float32), hi_r.astype(np.float32)
    return np.maximum(o - np.float32(B), lo_r).astype(np.float32), np.minimum(o + np.float32(B), hi_r).astype(np.float32)

def candidates(xc, X0c, j, T, B, K):
    """candidate (row, value) moves for feature j: the K nearest thresholds on each side inside the allowed interval, plus the interval ends."""
    xv = xc[:, j]; a, b = interval(X0c[:, j], j, B); nT = len(T); idx = np.searchsorted(T, xv, side='right'); R, V = [], []
    for d in range(min(K, nT)):
        k = idx + d; v = T[np.minimum(k, nT - 1)]; ok = (k < nT) & (v <= b); R.append(np.flatnonzero(ok)); V.append(v[ok])
        k = idx - 1 - d; v = np.nextafter(T[np.maximum(k, 0)], np.float32(-np.inf)); ok = (k >= 0) & (v >= a); R.append(np.flatnonzero(ok)); V.append(v[ok])
    ok = b > xv; R.append(np.flatnonzero(ok)); V.append(b[ok]); ok = a < xv; R.append(np.flatnonzero(ok)); V.append(a[ok])
    return np.concatenate(R), np.concatenate(V).astype(np.float32)

def one_step(margin, xc, X0c, g, feats, TH, B, K):
    """best single-feature move per row (by the model's own margin gap). Returns best_gap, best_feature (-1 = none helps), best_value, n_scored."""
    n = len(xc); best_g = g.copy(); best_j = np.full(n, -1, np.int64); best_v = np.zeros(n, np.float32); n_scored = 0
    bufR, bufJ, bufV = [], [], []; size = 0
    def flush():
        nonlocal n_scored
        if not bufR: return
        R, J, V = np.concatenate(bufR), np.concatenate(bufJ), np.concatenate(bufV); bufR.clear(); bufJ.clear(); bufV.clear()
        for s in range(0, len(R), BLOCK):
            r, jj, v = R[s:s + BLOCK], J[s:s + BLOCK], V[s:s + BLOCK]
            Xc = xc[r]; Xc[np.arange(len(r)), jj] = v                    # xc[r] is a copy
            gg = gap_of(margin(Xc)); n_scored += len(r)
            o = np.argsort(r, kind='stable'); rs, gs = r[o], gg[o]
            starts = np.flatnonzero(np.r_[True, rs[1:] != rs[:-1]]); segmax = np.maximum.reduceat(gs, starts); cnt = np.diff(np.r_[starts, len(rs)])
            hit = np.flatnonzero(gs == np.repeat(segmax, cnt)); _, first = np.unique(rs[hit], return_index=True); sel = o[hit[first]]
            rows = r[sel]; better = gg[sel] > best_g[rows]; rr = rows[better]
            best_g[rr] = gg[sel][better]; best_j[rr] = jj[sel][better]; best_v[rr] = v[sel][better]
    for j in feats:
        T = TH.get(int(j))
        if T is None: continue
        R, V = candidates(xc, X0c, int(j), T, B, K)
        if len(R) == 0: continue
        bufR.append(R); bufJ.append(np.full(len(R), j, np.int64)); bufV.append(V); size += len(R)
        if size >= BLOCK: flush(); size = 0
    flush(); return best_g, best_j, best_v, n_scored

def whitebox_trace(margin, X0, y0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):
    """as NB3c's whitebox, plus a trace: per round, the evaded share per class (classes 4, 5) and the median margin gap of the rows not yet evaded."""
    x = X0.copy(); g = gap_of(margin(x)); active = g <= 0; stalled = np.zeros(len(x), bool); trace = []; t_start = time.time()
    active = ~(g > 0); init_ev = g > 0
    for it in range(max_iters):
        act = np.flatnonzero(active)
        if len(act) == 0: break
        t0 = time.time(); scored = 0
        for s in range(0, len(act), chunk):
            ids = act[s:s + chunk]; bg, bj, bv, ns = one_step(margin, x[ids], X0[ids], g[ids], feats, TH, B, K); scored += ns
            imp = bj >= 0; ii = ids[imp]; x[ii, bj[imp]] = bv[imp]; g[ii] = bg[imp]; stalled[ids[~imp]] = True; active[ids[~imp]] = False
        active &= (g <= 0); ev = g > 0
        trace.append({'round': it + 1, 'active': int(len(act)), 'evaded_all': float(ev.mean()), 'evaded_pc': [float(ev[y0 == c].mean()) for c in range(1, 6)], 'evaded_c4': float(ev[y0 == 4].mean()), 'evaded_c5': float(ev[y0 == 5].mean()),
                      'median_gap_unevaded': float(np.median(g[~ev])) if (~ev).any() else float('nan'), 'seconds': time.time() - t0})
        if verbose and ((it + 1) % 10 == 0 or it == 0): print(f'      round {it + 1:>3}: active {len(act):>5} | evaded ransomware {trace[-1]["evaded_c4"]:.3f} reverseshell {trace[-1]["evaded_c5"]:.3f} | median gap of unevaded {trace[-1]["median_gap_unevaded"]:+.3f} | {time.time() - t0:4.1f}s', flush=True)
    return x, g, init_ev, stalled, trace
def whitebox(margin, X0, feats, TH, B, K=K_CAND, max_iters=MAX_ITERS, chunk=CHUNK, verbose=True):      # used by the self-test
    return whitebox_trace(margin, X0, np.zeros(len(X0), dtype=np.int64), feats, TH, B, K, max_iters, chunk, verbose)
print('white-box attack defined')

## 3. Gate and joint score (NB6 code)

In [ ]:
class AE(nn.Module):
    def __init__(self, n_features, bottleneck=GATE_BOTTLENECK, hidden=GATE_HIDDEN):
        super().__init__()
        self.encoder = nn.Sequential(nn.Linear(n_features, hidden), nn.GELU(), nn.Linear(hidden, bottleneck), nn.GELU())
        self.decoder = nn.Sequential(nn.Linear(bottleneck, hidden), nn.GELU(), nn.Linear(hidden, n_features))
    def forward(self, x): return self.decoder(self.encoder(x))
FW = torch.ones(N_FEAT, device=DEVICE); FW[torch.as_tensor(CYBER_IDX, device=DEVICE)] = GATE_CYBER_BOOST
def wmse(recon, target): return (((recon - target) ** 2) * FW).sum(1) / FW.sum()

class Gate:
    def __init__(self, models, tau): self.models = models; self.tau = float(tau)
    @torch.no_grad()
    def score(self, X, bs=65536):
        X = np.ascontiguousarray(X, dtype=np.float32); out = np.empty(len(X), np.float64)
        for s in range(0, len(X), bs):
            b = torch.from_numpy(X[s:s + bs]).to(DEVICE); out[s:s + bs] = torch.stack([wmse(m(b), b) for m in self.models]).mean(0).double().cpu().numpy()
        return out
    def flag(self, X): return self.score(X) > self.tau

def train_gate(seed, X_benign):
    from sklearn.model_selection import KFold
    oof = np.zeros(len(X_benign)); models = []
    for k, (tr, ho) in enumerate(KFold(N_FOLDS, shuffle=True, random_state=seed).split(X_benign)):
        torch.manual_seed(seed * 10 + k); m = AE(N_FEAT).to(DEVICE); opt = torch.optim.Adam(m.parameters(), lr=1e-3)
        ds = TensorDataset(torch.from_numpy(X_benign[tr]).float()); loader = DataLoader(ds, batch_size=256, shuffle=True, generator=torch.Generator().manual_seed(seed * 10 + k))
        for ep in range(GATE_EPOCHS):
            m.train()
            for (xb,) in loader: xb = xb.to(DEVICE); opt.zero_grad(); wmse(m(xb), xb).mean().backward(); opt.step()
        m.eval(); models.append(m); oof[ho] = Gate([m], 0).score(X_benign[ho])
    return Gate(models, np.percentile(oof, GATE_PCTL)), oof

class PipelineOracle:
    """hard-label verdict of the whole pipeline: a row Layer 1 calls benign but the gate flags is reported as an attack (class 1)."""
    def __init__(self, clf, gate): self.clf = clf; self.gate = gate; self.n_queries = 0
    def predict(self, X):
        X = np.ascontiguousarray(X, dtype=np.float32); self.n_queries += len(X); p = self.clf.predict(X)
        return np.where((p == 0) & self.gate.flag(X), 1, p)

class PipelineMargin:
    """the quantity the fully adaptive white-box attacker maximises: min(classifier margin gap, GATE_SCALE * (tau - error) / tau); > 0 <=> evades both layers.
    Returned as a 2-column array so that the existing gap_of(z) = z[:,0] - max(z[:,1:]) equals this score."""
    def __init__(self, clf_gpu, gate): self.m = Margin(clf_gpu); self.gate = gate
    def __call__(self, X):
        s = np.minimum(gap_of(self.m(X)), GATE_SCALE * (self.gate.tau - self.gate.score(X)) / self.gate.tau)
        return np.stack([s, np.zeros_like(s)], 1)
print('gate, pipeline oracle and joint-score margin defined')

## 4. Rows

In [ ]:
rng0 = np.random.RandomState(0); pick = np.sort(np.concatenate([rng0.choice(np.flatnonzero(y_atk_test == c), N_WB_PER_CLASS, replace=False) for c in range(1, 6)]))
X_wb, y_wb = X_atk_test[pick], y_atk_test[pick]                           # NB5 / NB5b / NB5c rows
rng1 = np.random.RandomState(1); sub = np.sort(np.concatenate([rng1.choice(np.flatnonzero(y_wb == c), N_AW_PER_CLASS, replace=False) for c in range(1, 6)]))   # random subset (not the earliest rows)
X_aw, y_aw = X_wb[sub], y_wb[sub]
def pcs(ev, y): return np.array([ev[y == c].mean() for c in range(1, 6)])
def load_model(path, seed):
    gpu = xgb.XGBClassifier(**best_params, **COMMON, random_state=seed); gpu.load_model(path)
    cpu = xgb.XGBClassifier(); cpu.load_model(path); cpu.set_params(device='cpu', n_jobs=-1); return gpu, cpu
print(f'attack rows used by the white-box attackers: {len(X_aw)} ({N_AW_PER_CLASS} per class) | budgets {AW_BUDGETS} | PGD / PPO use all {len(X_atk_test):,} test attack rows')

In [ ]:
rng2 = np.random.RandomState(2); sub2 = np.sort(np.concatenate([rng2.choice(np.flatnonzero(y_aw == c), N_LONG_PER_CLASS, replace=False) for c in range(1, 6)])); X_lg, y_lg = X_aw[sub2], y_aw[sub2]
print(f'rows for the long attack: {len(X_lg)} ({N_LONG_PER_CLASS} per class) | budgets {LONG_BUDGETS} | up to {LONG_ITERS} rounds')
def verdict(clf_cpu, gate, X):
    ev1 = clf_cpu.predict(X) == 0; fl = gate.flag(X); return ev1, fl, ev1 & ~fl
def load_gate(seed):
    ck = torch.load(f'ae_gate_{PIPELINE_VERSION}_seed{seed}.pt', map_location=DEVICE); ms_ = []
    for st in ck['state']: m = AE(N_FEAT).to(DEVICE); m.load_state_dict(st); m.eval(); ms_.append(m)
    return Gate(ms_, ck['tau'])

## 5. Per-seed run

In [ ]:
R = {}; t_all = time.time()
for seed in SEEDS:
    pf = f'attack3b_seed{seed}_{PIPELINE_VERSION}.pkl'
    cfg_fp = repr((LAYER1, LONG_BUDGETS, LONG_ITERS, CHECKPOINTS, N_LONG_PER_CLASS, N_AW_PER_CLASS, GATE_SCALE, K_CAND, seed, X_atk_test.shape, float(X_atk_test[:5].sum()), 'nb6b-v1'))
    if os.path.exists(pf):
        saved = joblib.load(pf)
        if saved.get('fp') == cfg_fp: R[seed] = saved['R']; print(f'seed {seed}: result file found -- skipped'); continue
    t0 = time.time(); print(f'=== seed {seed}', flush=True); r = {}
    gate = load_gate(seed); nb6 = joblib.load(f'attack3_seed{seed}_{PIPELINE_VERSION}.pkl')['R']; sc = gate.score(X_test)
    r['gate_check'] = {'tau': gate.tau, 'fpr_now': float((sc[y_test == 0] > gate.tau).mean()), 'fpr_nb6': float(nb6['gate_clean']['fpr_test'])}
    print(f'  gate reloaded: tau {gate.tau:.5f}, clean FPR {r["gate_check"]["fpr_now"]:.4f} (NB6 {r["gate_check"]["fpr_nb6"]:.4f})', flush=True); assert abs(r['gate_check']['fpr_now'] - r['gate_check']['fpr_nb6']) < 1e-3
    for mtag, pat in LAYER1.items():
        gpu_m, cpu_m = load_model(pat.format(v=PIPELINE_VERSION, s=seed), seed); TH = get_thresholds(gpu_m); pm = PipelineMargin(gpu_m, gate)
        for bt, B in LONG_BUDGETS.items():
            ts = time.time(); xa, ga, ie, st, tr = whitebox_trace(pm, X_lg, y_lg, CYBER_IDX, TH, B, max_iters=LONG_ITERS, verbose=False); ev1, fl, evp = verdict(cpu_m, gate, xa)
            curve = {c: tr[min(c, len(tr)) - 1]['evaded_all'] for c in CHECKPOINTS}; curve_pc = {c: np.array(tr[min(c, len(tr)) - 1]['evaded_pc']) for c in CHECKPOINTS}
            r[f'{mtag}|{bt}'] = {'pipeline_evasion': float(evp.mean()), 'pipeline_pc': pcs(evp, y_lg), 'l1_evasion': float(ev1.mean()), 'curve': curve, 'curve_pc': curve_pc, 'rounds_run': len(tr), 'still_improving': float((~st & ~evp).mean()),
                                 'still_improving_pc': pcs(~st & ~evp, y_lg) if False else np.array([(~st & ~evp)[y_lg == c].mean() for c in range(1, 6)]), 'stalled': float((st & ~evp).mean()), 'search_vs_cpu_disagree': float(((ga > 0) != evp).mean())}
            print(f'  [{mtag}] {bt} sigma: pipeline evasion @100 {curve[100]:.3f} -> final {evp.mean():.3f} | per-class {np.round(pcs(evp, y_lg), 3).tolist()} | still improving {r[f"{mtag}|{bt}"]["still_improving"]:.3f} (per class {np.round(r[f"{mtag}|{bt}"]["still_improving_pc"], 2).tolist()}) | rounds {len(tr)} | {(time.time() - ts) / 60:.1f} min', flush=True)
        del gpu_m, cpu_m, pm; torch.cuda.empty_cache()
    R[seed] = r; joblib.dump({'fp': cfg_fp, 'R': r}, pf); print(f'  seed {seed} done in {(time.time() - t0) / 60:.1f} min', flush=True)
print(f'Total {(time.time() - t_all) / 60:.1f} min')

## 6. Results

In [ ]:
atk_names = class_names[1:]
V = lambda key, f: np.array([R[s][key][f] for s in SEEDS]); ms = lambda x: f'{np.mean(x):.3f} +/- {(np.std(x, ddof=1) if len(x) > 1 else 0):.3f}'
print('6a. Reloaded gates reproduce NB6:', {s: (round(R[s]['gate_check']['fpr_now'], 4), round(R[s]['gate_check']['fpr_nb6'], 4)) for s in SEEDS})
rows = []
for m in LAYER1:
    for bt in LONG_BUDGETS:
        k = f'{m}|{bt}'; cur = {c: np.array([R[s][k]['curve'][c] for s in SEEDS]) for c in CHECKPOINTS}; fin = V(k, 'pipeline_evasion')
        rows.append({'Layer 1': m, 'budget': f'{bt} sigma', **{f'@{c}': f'{cur[c].mean():.3f}' for c in CHECKPOINTS}, 'final (CPU)': ms(fin), 'gain 100 -> final': f'{(fin - cur[100]).mean():+.4f}', 'still improving': f'{V(k, "still_improving").mean():.3f}', 'rounds run (max)': int(max(R[s][k]["rounds_run"] for s in SEEDS))})
conv = pd.DataFrame(rows); pd.set_option('display.width', 320); print('\n6b. Pipeline evasion of the fully adaptive white-box vs rounds (mean over seeds)'); print(conv.to_string(index=False))
prow = []
for m in LAYER1:
    for bt in LONG_BUDGETS:
        k = f'{m}|{bt}'; c100 = np.stack([R[s][k]['curve_pc'][100] for s in SEEDS]); fin = np.stack([R[s][k]['pipeline_pc'] for s in SEEDS]); si = np.stack([R[s][k]['still_improving_pc'] for s in SEEDS])
        prow.append({'Layer 1': m, 'budget': f'{bt} sigma', **{c: f'{c100[:, i].mean():.3f} -> {fin[:, i].mean():.3f} ({si[:, i].mean():.2f})' for i, c in enumerate(atk_names)}})
pct = pd.DataFrame(prow); print('\n6c. Per-class pipeline evasion: @100 rounds -> final (share of that class still improving at the end)'); print(pct.to_string(index=False))
print('\n6d. Ransomware / reverseshell verdict (pipeline evasion after the long attack; per-seed values)')
for m in LAYER1:
    for bt in LONG_BUDGETS:
        k = f'{m}|{bt}'; fin = np.stack([R[s][k]['pipeline_pc'] for s in SEEDS]); si = np.stack([R[s][k]['still_improving_pc'] for s in SEEDS])
        print(f'   [{m}] {bt} sigma: ransomware {np.round(fin[:, 3], 3).tolist()} reverseshell {np.round(fin[:, 4], 3).tolist()} | still improving ransomware {si[:, 3].mean():.3f} reverseshell {si[:, 4].mean():.3f} -> ' + ('CONVERGED' if si[:, 3:].max() < 0.01 else 'NOT converged within the round limit'))
print('\n6e. Search/CPU disagreement:', {f'{m}|{bt}': round(float(V(f"{m}|{bt}", "search_vs_cpu_disagree").mean()), 5) for m in LAYER1 for bt in LONG_BUDGETS})
nb6c = {}
for m in LAYER1:
    k = f'{m}|wb_aware_1.0'
    try: nb6c[m] = np.mean([joblib.load(f'attack3_seed{s}_{PIPELINE_VERSION}.pkl')['R'][k]['pipeline_evasion'] for s in SEEDS])
    except Exception: pass
print('\n6f. For reference, NB6 @100 rounds, 1.0 sigma, 1,000 rows per class (different subset):', {m: round(float(v), 3) for m, v in nb6c.items()})

## 7. Figure and save

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(15, 4.8))
for m, a in zip(LAYER1, ax):
    for bt in LONG_BUDGETS:
        k = f'{m}|{bt}'
        for ci, cn in enumerate(atk_names):
            v = np.array([[R[s][k]['curve_pc'][c][ci] for c in CHECKPOINTS] for s in SEEDS]).mean(0); a.plot(CHECKPOINTS, v, ('-' if bt == '1.0' else '--'), marker='o', color=f'C{ci}', label=f'{cn} {bt}' if True else None)
    a.set_xlabel('white-box rounds'); a.set_ylabel('pipeline evasion'); a.set_title(f'Fully adaptive white-box, Layer 1 = {m} (solid 1.0 sigma, dashed 0.5 sigma)'); a.grid(alpha=.3); a.legend(fontsize=6, ncol=2)
plt.tight_layout(); plt.savefig(f'attack3b_convergence_{PIPELINE_VERSION}.png', dpi=150); plt.show()
joblib.dump({'PIPELINE_VERSION': PIPELINE_VERSION, 'SEEDS': SEEDS, 'conv_table': conv, 'per_class_table': pct, 'per_seed': {s: R[s] for s in SEEDS}, 'config': {'LONG_ITERS': LONG_ITERS, 'CHECKPOINTS': CHECKPOINTS, 'LONG_BUDGETS': LONG_BUDGETS, 'N_LONG_PER_CLASS': N_LONG_PER_CLASS}}, f'attack3b_results_{PIPELINE_VERSION}.pkl')
conv.to_csv(f'attack3b_convergence_table_{PIPELINE_VERSION}.csv', index=False); pct.to_csv(f'attack3b_per_class_{PIPELINE_VERSION}.csv', index=False)
json.dump({'notebook': 'NB6b', 'pipeline_version': PIPELINE_VERSION, 'seeds': SEEDS, 'xgboost': xgb.__version__, 'utc': time.strftime('%Y-%m-%d %H:%M:%S', time.gmtime())}, open(f'nb6b_run_manifest_{PIPELINE_VERSION}.json', 'w'), indent=2)
for f in [f'attack3b_results_{PIPELINE_VERSION}.pkl', f'attack3b_convergence_table_{PIPELINE_VERSION}.csv', f'attack3b_per_class_{PIPELINE_VERSION}.csv', f'attack3b_convergence_{PIPELINE_VERSION}.png', f'nb6b_run_manifest_{PIPELINE_VERSION}.json']: print(('  OK      ' if os.path.exists(f) else '  MISSING ') + f)